In [0]:
-- ============================================================
-- SILVER LAYER: CLEANED AND VALIDATED DATA
-- Source: bronze_oneplus_products
-- Purpose:
--   1. Validate OnePlus product and SKU records
--   2. Correct price data types
--   3. Standardize stock status
--   4. Calculate discount amount
--   5. Add data quality flag
-- ============================================================

CREATE OR REFRESH STREAMING TABLE silver_oneplus_products (

  -- Data quality constraints

  CONSTRAINT valid_product_code
    EXPECT (product_code IS NOT NULL)
    ON VIOLATION DROP ROW,

  CONSTRAINT valid_product_name
    EXPECT (product_name IS NOT NULL)
    ON VIOLATION DROP ROW,

  CONSTRAINT valid_sku_code
    EXPECT (sku_code IS NOT NULL)
    ON VIOLATION DROP ROW,

  CONSTRAINT valid_category
    EXPECT (category_name IS NOT NULL)
    ON VIOLATION DROP ROW,

  CONSTRAINT valid_price
    EXPECT (
      original_price >= 0
      AND now_price >= 0
    )
    ON VIOLATION DROP ROW

)

COMMENT "Silver layer: Cleaned and validated OnePlus product and SKU data"

AS

SELECT

  -- ==========================================================
  -- CATEGORY
  -- ==========================================================

  category_name,
  category_code,
  category_url,

  -- ==========================================================
  -- PRODUCT
  -- ==========================================================

  product_name,
  product_code,
  product_url,
  product_short_url,

  -- ==========================================================
  -- SKU
  -- ==========================================================

  sku_code,
  sku_name,

  -- ==========================================================
  -- PRODUCT ATTRIBUTES
  -- ==========================================================

  color,
  color_code,
  ram_storage,
  storage_code,

  -- ==========================================================
  -- PRICE
  -- ==========================================================

  CAST(original_price AS DECIMAL(12,2))
    AS original_price,

  CAST(sale_price AS DECIMAL(12,2))
    AS sale_price,

  CAST(now_price AS DECIMAL(12,2))
    AS now_price,

  CAST(discount_rate AS DECIMAL(8,2))
    AS discount_rate,

  discount_tag,

  -- ==========================================================
  -- STOCK
  -- ==========================================================

  CAST(has_stock AS BOOLEAN)
    AS has_stock,

  CAST(out_stock AS BOOLEAN)
    AS out_stock,

  CAST(low_stock AS BOOLEAN)
    AS low_stock,

  -- ==========================================================
  -- PRODUCT STATUS
  -- ==========================================================

  CAST(new_product AS BOOLEAN)
    AS new_product,

  sale_status,

  CAST(has_gift AS BOOLEAN)
    AS has_gift,

  -- ==========================================================
  -- IMAGE
  -- ==========================================================

  image_url,
  image_alt,

  -- ==========================================================
  -- CALCULATED FIELD: DISCOUNT AMOUNT
  -- ==========================================================

  ROUND(
    CAST(original_price AS DECIMAL(12,2))
    -
    CAST(now_price AS DECIMAL(12,2)),
    2
  ) AS discount_amount,

  -- ==========================================================
  -- CALCULATED FIELD: STOCK STATUS
  -- ==========================================================

  CASE
    WHEN CAST(out_stock AS BOOLEAN) = TRUE
      THEN 'Out of Stock'

    WHEN CAST(low_stock AS BOOLEAN) = TRUE
      THEN 'Low Stock'

    WHEN CAST(has_stock AS BOOLEAN) = TRUE
      THEN 'In Stock'

    ELSE 'Unknown'
  END AS stock_status,

  -- ==========================================================
  -- DATA QUALITY FLAG
  -- ==========================================================

  CASE

    WHEN product_code IS NULL
      THEN 'MISSING_PRODUCT_CODE'

    WHEN product_name IS NULL
      THEN 'MISSING_PRODUCT_NAME'

    WHEN sku_code IS NULL
      THEN 'MISSING_SKU_CODE'

    WHEN original_price < 0
      THEN 'INVALID_ORIGINAL_PRICE'

    WHEN now_price < 0
      THEN 'INVALID_CURRENT_PRICE'

    WHEN original_price < now_price
      THEN 'PRICE_INCREASE'

    ELSE 'VALID'

  END AS data_quality_flag,

  -- ==========================================================
  -- INGESTION
  -- ==========================================================

  ingestion_timestamp

FROM STREAM(
  bronze_oneplus_products
);


SELECT COUNT(*) AS total_rows
FROM silver_oneplus_products;


SELECT *
FROM silver_oneplus_products
LIMIT 10;

SELECT
  data_quality_flag,
  COUNT(*) AS record_count
FROM silver_oneplus_products
GROUP BY data_quality_flag;

SELECT
  stock_status,
  COUNT(*) AS sku_count
FROM silver_oneplus_products
GROUP BY stock_status;

SELECT
  MIN(now_price) AS minimum_price,
  MAX(now_price) AS maximum_price,
  ROUND(AVG(now_price), 2) AS average_price
FROM silver_oneplus_products;